<a href="https://colab.research.google.com/github/Jeeshau/material-intelligence-engine/blob/main/material_engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Phase 1

Designed a relational database schema and ingested 174 real project material records from Notion into SQLite, with automated data cleaning for price normalization and null handling

In [1]:
# Cell 1 — 连接 Google Drive + 上传数据
from google.colab import drive, files
drive.mount('/content/drive')

import pandas as pd

# 上传你的两个 CSV
uploaded = files.upload()


Mounted at /content/drive


Saving GOODVETS - ALL LOCATIONS - FFE 2023 - 540 HUDSON.csv to GOODVETS - ALL LOCATIONS - FFE 2023 - 540 HUDSON.csv
Saving Alidoro Master Database 2df80731963d8130bd80ec263c81fd82_all.csv to Alidoro Master Database 2df80731963d8130bd80ec263c81fd82_all.csv


In [2]:
# Cell 2 — 读取并预览数据
import pandas as pd

# 读取 Notion 数据 (Alidoro)
notion_df = pd.read_csv('Alidoro Master Database 2df80731963d8130bd80ec263c81fd82_all.csv')

# 读取 Google Sheet 数据 (Goodvets) - 跳过前几行header
google_df = pd.read_csv('GOODVETS - ALL LOCATIONS - FFE 2023 - 540 HUDSON.csv', skiprows=5, header=1)

print("=== Notion 数据 ===")
print(f"行数: {len(notion_df)}, 列数: {len(notion_df.columns)}")
print(notion_df[['Name', 'Category (Designer Use)', 'Manufacturer', 'Unit Cost', 'Lead Time', 'Order Status']].head(5))

print("\n=== Google Sheet 数据 ===")
print(f"行数: {len(google_df)}, 列数: {len(google_df.columns)}")
print(google_df.head(5))

=== Notion 数据 ===
行数: 174, 列数: 47
                    Name Category (Designer Use)   Manufacturer   Unit Cost  \
0               Recessed        Lighting Fixture           Lusa     $80.10    
1  Quarry Tile Cove Base                  Finish        Tilebar      $0.00    
2                  Stool               Furniture  Anthropologie    $358.20    
3           Zellige Tile                  Finish       Zia Tile  $2,983.28    
4                 Toilet        Plumbing Fixture            NaN         NaN   

           Lead Time  Order Status  
0  3-5 Business Days      ESTIMATE  
1           In Stock     UPDATE PO  
2                NaN  NOT IN SCOPE  
3           In Stock     UPDATE PO  
4                NaN  NOT IN SCOPE  

=== Google Sheet 数据 ===
行数: 66, 列数: 16
                                          Unnamed: 0   PT-01  \
0                                                NaN   PT-02   
1  FOR DOORS I'D\nGO WITH LACQUER\nOR LAMINATE\nF...   PT-03   
2                                    

In [3]:
# Cell 3 — 清洗 Notion 数据 + 导入 SQLite

import sqlite3
import pandas as pd
import re

# 1. 选取核心列
notion_clean = notion_df[[
    'Name', 'Category (Designer Use)', 'Manufacturer', 'Model',
    'Description', 'Finish / Color', 'Product Size', 'Unit',
    'Unit Cost', 'Lead Time', 'Order Status', 'Approval Status',
    'TAG', 'Link', 'Location', 'NOTES'
]].copy()

# 2. 重命名列（去掉空格和括号，SQL友好）
notion_clean.columns = [
    'name', 'category', 'manufacturer', 'model',
    'description', 'finish_color', 'product_size', 'unit',
    'unit_cost', 'lead_time', 'order_status', 'approval_status',
    'tag', 'link', 'location', 'notes'
]

# 3. 清洗 unit_cost：把 "$1,121.40" 变成数字 1121.40
def clean_price(val):
    if pd.isna(val):
        return None
    cleaned = re.sub(r'[$,]', '', str(val)).strip()
    try:
        return float(cleaned)
    except:
        return None

notion_clean['unit_cost'] = notion_clean['unit_cost'].apply(clean_price)

# 4. 清洗空白字符串
notion_clean = notion_clean.replace('', None)
notion_clean['name'] = notion_clean['name'].str.strip()
notion_clean['category'] = notion_clean['category'].str.strip()

# 5. 加项目来源标记
notion_clean['source'] = 'Notion'
notion_clean['project'] = 'Alidoro ESB'

# 6. 导入 SQLite
conn = sqlite3.connect('materials.db')
notion_clean.to_sql('materials', conn, if_exists='replace', index=True, index_label='id')
conn.close()

print(f"✅ 成功导入 {len(notion_clean)} 条材料数据")
print("\n预览清洗后的数据：")
print(notion_clean[['name', 'category', 'manufacturer', 'unit_cost', 'order_status']].head(8).to_string())


✅ 成功导入 174 条材料数据

预览清洗后的数据：
                        name          category             manufacturer  unit_cost  order_status
0                   Recessed  Lighting Fixture                     Lusa      80.10      ESTIMATE
1      Quarry Tile Cove Base            Finish                  Tilebar       0.00     UPDATE PO
2                      Stool         Furniture            Anthropologie     358.20  NOT IN SCOPE
3               Zellige Tile            Finish                 Zia Tile    2983.28     UPDATE PO
4                     Toilet  Plumbing Fixture                      NaN        NaN  NOT IN SCOPE
5                    Troffer  Lighting Fixture                    TOPAZ      32.13      ESTIMATE
6  Track Head\nSurface Mount  Lighting Fixture  Etsy_\nSmile Lamp Works     169.00      ESTIMATE
7   Eliason - Single Traffic              Door                      NaN    2700.00      ESTIMATE


In [4]:
# Cell 4 — 你的第一批 SQL 查询

import sqlite3
import pandas as pd

conn = sqlite3.connect('materials.db')

# 查询 1: 所有材料类别有哪些？
print("=== 材料类别分布 ===")
q1 = pd.read_sql_query("""
    SELECT category, COUNT(*) as 数量
    FROM materials
    WHERE category IS NOT NULL
    GROUP BY category
    ORDER BY 数量 DESC
""", conn)
print(q1.to_string())

# 查询 2: 所有 Lighting Fixture，按价格排序
print("\n=== Lighting Fixtures (按价格) ===")
q2 = pd.read_sql_query("""
    SELECT name, manufacturer, unit_cost, order_status
    FROM materials
    WHERE category = 'Lighting Fixture'
    AND unit_cost IS NOT NULL
    ORDER BY unit_cost DESC
""", conn)
print(q2.to_string())

# 查询 3: 还没有下单的材料 (ESTIMATE状态)
print("\n=== 待确认材料 (ESTIMATE) ===")
q3 = pd.read_sql_query("""
    SELECT name, category, manufacturer, unit_cost
    FROM materials
    WHERE order_status = 'ESTIMATE'
    AND unit_cost IS NOT NULL
    ORDER BY unit_cost DESC
    LIMIT 10
""", conn)
print(q3.to_string())

conn.close()

=== 材料类别分布 ===
                category  数量
0       Millwork Catalog  88
1                 Finish  34
2               Hardware  25
3       Lighting Fixture  16
4              Furniture   4
5                   Door   2
6       Plumbing Fixture   1
7  Drafting Place Holder   1

=== Lighting Fixtures (按价格) ===
                         name                 manufacturer  unit_cost order_status
0              Custom Pendant                       Custom    2500.00     ESTIMATE
1                 Wall Sconce                   Slow Roads     378.00     ESTIMATE
2              Linear Pendant  Etsy_\nNew Wine Old Bottles     375.00     ESTIMATE
3   Track Head\nSurface Mount      Etsy_\nSmile Lamp Works     169.00     ESTIMATE
4                         LED                   ASPECT LED     156.50     ESTIMATE
5                     Pendant               Ideas4Lighting     116.37     ESTIMATE
6               New Exit Sign                       AtLite     100.00     ESTIMATE
7                    Recess

## Phase 2 — ETL Pipeline

Built an ETL pipeline ingesting material data from two sources (Notion API export + Google Sheets), normalizing heterogeneous schemas into a unified SQLite database across 227 records from 2 real interior design projects

In [5]:
# Cell 5 — 清洗 Google Sheet 数据 + 合并进数据库

import sqlite3
import pandas as pd
import re

# 重新读取 Google Sheet，找到真正的数据行
raw = pd.read_csv('GOODVETS - ALL LOCATIONS - FFE 2023 - 540 HUDSON.csv',
                  header=None)

# 找到包含 TAG 列的那一行作为 header
header_row = None
for i, row in raw.iterrows():
    if any(str(v).strip().upper() in ['TAG', 'PT-01', 'ITEM']
           for v in row.values if pd.notna(v)):
        header_row = i
        break

print(f"Header 在第 {header_row} 行")
print(raw.iloc[header_row].tolist())

Header 在第 5 行
['STATUS', 'TAG', 'AREA', 'IMAGE', 'ITEM DESCRIPTION', 'Qty. (10%)', nan, 'UNIT PRICE', 'CUSTOMIZATION CHARGE', 'SHIPPING / PACKING', 'TAX', 'TOTAL PRICE \n(SHIP&TAX)', 'LEAD TIME', 'WEBSITE', 'SPEC', 'CONTACT INFO']


In [6]:
# Cell 6 — 清洗 Google Sheet + 合并进数据库

import sqlite3
import pandas as pd
import re

# 用第5行作为 header 重新读取
gs_df = pd.read_csv('GOODVETS - ALL LOCATIONS - FFE 2023 - 540 HUDSON.csv',
                    skiprows=5, header=0)

print("原始列名:", gs_df.columns.tolist())
print(f"行数: {len(gs_df)}")
print(gs_df.head(3).to_string())

原始列名: ['STATUS', 'TAG', 'AREA', 'IMAGE', 'ITEM DESCRIPTION', 'Qty. (10%)', 'Unnamed: 6', 'UNIT PRICE', 'CUSTOMIZATION CHARGE', 'SHIPPING / PACKING', 'TAX', 'TOTAL PRICE \n(SHIP&TAX)', 'LEAD TIME', 'WEBSITE', 'SPEC', 'CONTACT INFO']
行数: 67
                                                                                                 STATUS    TAG            AREA  IMAGE                                                                                                              ITEM DESCRIPTION  Qty. (10%) Unnamed: 6 UNIT PRICE  CUSTOMIZATION CHARGE SHIPPING / PACKING    TAX TOTAL PRICE \n(SHIP&TAX)                      LEAD TIME WEBSITE SPEC CONTACT INFO
0                                                                                                   NaN  PT-01     WALLS\nTYP.    NaN                                          INTERIOR / Eggshell Finish\nBENJAMIN MOORE WHITE OC-151\nSTANDARD, COMMERCIAL, GRADE         NaN     GALLON     $82.99                   NaN                NaN  $

In [7]:
# Cell 7 — 完整清洗 Google Sheet + 合并入数据库

def clean_price(val):
    if pd.isna(val): return None
    cleaned = re.sub(r'[$,]', '', str(val)).strip()
    try: return float(cleaned)
    except: return None

# 选取有用的列
gs_clean = gs_df[[
    'STATUS', 'TAG', 'AREA', 'ITEM DESCRIPTION',
    'UNIT PRICE', 'Qty. (10%)', 'SHIPPING / PACKING'
]].copy()

# 重命名对齐 Notion 的格式
gs_clean.columns = [
    'order_status', 'tag', 'location', 'name',
    'unit_cost', 'unit', 'notes'
]

# 清洗
gs_clean['unit_cost'] = gs_clean['unit_cost'].apply(clean_price)
gs_clean['name'] = gs_clean['name'].str.replace(r'\n', ' ', regex=True).str.strip()
gs_clean['location'] = gs_clean['location'].str.replace(r'\n', ' ', regex=True).str.strip()

# 补充空列（Notion有但Google Sheet没有的）
gs_clean['category'] = None
gs_clean['manufacturer'] = None
gs_clean['model'] = None
gs_clean['description'] = gs_clean['name']
gs_clean['finish_color'] = None
gs_clean['product_size'] = None
gs_clean['lead_time'] = None
gs_clean['approval_status'] = gs_clean['order_status']
gs_clean['link'] = None
gs_clean['source'] = 'Google Sheet'
gs_clean['project'] = 'Goodvets 540 Hudson'

# 去掉 name 为空的行
gs_clean = gs_clean.dropna(subset=['name'])
gs_clean = gs_clean[gs_clean['name'].str.len() > 2]

# 合并进数据库
conn = sqlite3.connect('materials.db')
gs_clean.to_sql('materials', conn, if_exists='append', index=True, index_label='id')

# 验证合并结果
total = pd.read_sql_query("SELECT COUNT(*) as total FROM materials", conn)
by_project = pd.read_sql_query("""
    SELECT project, COUNT(*) as 数量
    FROM materials
    GROUP BY project
""", conn)
conn.close()

print(f"✅ Google Sheet 导入 {len(gs_clean)} 条")
print(f"\n=== 数据库总览 ===")
print(total.to_string())
print("\n=== 按项目分布 ===")
print(by_project.to_string())

✅ Google Sheet 导入 53 条

=== 数据库总览 ===
   total
0    227

=== 按项目分布 ===
               project   数量
0          Alidoro ESB  174
1  Goodvets 540 Hudson   53
